# Calendar Agent — Colab LoRA 학습 (단일 GPU · bf16 · HF push)

**준비**
1. 런타임: 상단 `런타임 → 런타임 유형 변경 → T4 GPU` 선택
2. GitHub PAT (repo scope): https://github.com/settings/tokens/new?type=classic
3. HuggingFace **write** 토큰: https://huggingface.co/settings/tokens

셀을 위→아래로 실행. 단일 T4 기준 0.5B LoRA ~1~1.5시간.
결과(어댑터)는 **학습 직후 HF Hub에 push**되어 세션이 죽어도 보존됨.

In [ ]:
# 1. GPU 확인 — Tesla T4 떠야 함. 안 뜨면 런타임 유형을 GPU로.
!nvidia-smi

In [ ]:
# 2. repo clone (private → PAT). 이미 있으면 origin/main으로 force-sync
import os, getpass
%cd /content
if not os.path.exists('calendar-agent'):
    token = getpass.getpass('GitHub PAT (repo scope): ').strip()
    !git clone https://{token}@github.com/sooryong/calendar-agent.git
    token = None
else:
    !cd calendar-agent && git fetch origin && git reset --hard origin/main
%cd /content/calendar-agent
!git log --oneline -1

In [ ]:
# 3. 설치 + 환경 (wandb off). torch가 +cpu면 GPU 미할당 → 런타임 GPU로 바꾸고 재시작
!pip install -q -e .[train]
!pip uninstall torchao -y -q   # PEFT가 거부하는 구버전 torchao 제거 (LoRA엔 불필요)
import os, torch
os.environ['WANDB_DISABLED'] = 'true'
print(f"torch={torch.__version__}  cuda={torch.cuda.is_available()}  "
      f"device={torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE'}")
assert torch.cuda.is_available(), "GPU 미할당 — 런타임 유형을 T4 GPU로 바꾸고 셀 재실행

In [ ]:
# 4. 데이터 확인 — golden 50건이어야 현 기준 평가
for p in ['data/processed/train.jsonl', 'data/processed/val.jsonl', 'data/eval/real_golden.jsonl']:
    n = sum(1 for l in open(p, encoding='utf-8') if l.strip())
    print(f'{p}: {n}')
n_gold = sum(1 for l in open('data/eval/real_golden.jsonl', encoding='utf-8') if l.strip())
assert n_gold == 50, f'golden {n_gold}건 (50 기대) — origin/main 동기화 확인'

In [ ]:
# 5. 학습 (단일 GPU · bf16 · configs/train.yaml = r27). torchrun 아님.
#    시작 로그에 '[train] 모델: Qwen/Qwen2.5-0.5B-Instruct' 확인. ~1~1.5h
!python scripts/train_lora.py --config configs/train.yaml

In [ ]:
# 6. merge + 골든 평가 (train.yaml output_dir에서 라운드 자동 인식)
import yaml, os
cfg = yaml.safe_load(open('configs/train.yaml'))
LORA_DIR = cfg['output_dir']                 # models/lora/r27-qwen0.5b
NAME     = os.path.basename(LORA_DIR)        # r27-qwen0.5b
MERGED   = f'models/merged/{NAME}'
EVAL_JSON= f'logs/eval_{NAME}.json'
print(f'round name={NAME}')
!python scripts/merge_lora.py --base Qwen/Qwen2.5-0.5B-Instruct --lora {LORA_DIR} --out {MERGED}
!python scripts/eval_model.py --model {MERGED} --eval data/eval/real_golden.jsonl --out {EVAL_JSON}
print(f'
--- {EVAL_JSON} ---')
print(open(EVAL_JSON, encoding='utf-8').read())

In [ ]:
# 7. HF Hub에 어댑터 push (세션 죽어도 보존). ★ HF_REPO를 본인 계정으로 바꾸세요.
import getpass
from huggingface_hub import login, create_repo, upload_folder, upload_file
HF_REPO = 'CHANGE_ME/calendar-lora'          # 예: 'sooryong/calendar-lora'
hf_token = getpass.getpass('HF write token: ').strip()
login(token=hf_token); hf_token = None
create_repo(HF_REPO, repo_type='model', exist_ok=True, private=True)
upload_folder(repo_id=HF_REPO, folder_path=LORA_DIR, path_in_repo=NAME, repo_type='model',
              commit_message=f'{NAME} LoRA adapter')
upload_file(repo_id=HF_REPO, path_or_fileobj=EVAL_JSON, path_in_repo=f'{NAME}/eval.json', repo_type='model')
print(f'pushed -> https://huggingface.co/{HF_REPO}/tree/main/{NAME}')

In [ ]:
# 8. (선택) 브라우저로도 어댑터 zip 받기 — HF push 했으면 생략 가능
ZIP = f'/content/lora_{NAME}.zip'
!zip -qr {ZIP} {LORA_DIR} {EVAL_JSON} configs/train.yaml configs/lora.yaml configs/model_qwen.yaml
from google.colab import files
files.download(ZIP)